RANDOM FOREST REGRESSOR MODEL

Michael Owens 

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [3]:
# Bringing in the Data

# read data
df = pd.read_csv('LassoFeatures.csv')
df = df.drop('Unnamed: 0',axis=1)
print(df.shape)
df.head(3)

(15351, 41)


,disc_year,sy_snum glon st_rad,sy_pnum glon ra,sy_pnum dec sy_pm,sy_pnum dec sy_plx,sy_pnum elat sy_pm,sy_pnum elat sy_plx,sy_pnum elat st_teff,sy_pnum pl_ntranspec st_rad,sy_pnum st_rad pl_orbper,...,disc_year st_rad pl_orbper,sy_vmag^2 st_rad,sy_vmag^2 st_mass,sy_vmag sy_bmag st_mass,sy_kmag st_teff pl_orbper,sy_gaiamag^3,st_rad sy_plx pl_orbper,st_rad st_mass soltype_Published Confirmed,st_teff^2 pl_orbper,log radius
0,0.132709,1.109439,-1.246369,0.072314,-0.212191,-0.668538,-0.633221,-1.490701,-0.140779,-0.107814,...,0.194529,-0.876743,-1.811754,-1.819948,0.099019,-2.185486,1.520130,1.821839,0.294147,0.372075
1,0.132709,1.038075,-1.246369,0.072314,-0.212191,-0.668538,-0.633221,-1.490521,-0.140779,-0.115598,...,0.175636,-0.907919,-1.819551,-1.828321,0.105602,-2.185486,1.463468,1.753110,0.311567,0.348305
2,2.065912,2.937174,1.971716,-0.623103,-0.582859,-0.635000,-0.578307,-1.425261,-0.140779,-0.360297,...,-0.432213,-0.649115,-1.186475,-1.218871,-0.487231,-1.785742,-0.369202,-0.306409,-0.476087,0.079181


Grid Search to Determine the Number of Estimators and Tree Depth

In [4]:
#Split data & define features/targets
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)
X_train = df_train.drop('log radius',axis=1)
X_test  = df_test.drop('log radius',axis=1)
y_train   = df_train['log radius']
y_test    = df_test['log radius']
X_train.describe()

,disc_year,sy_snum glon st_rad,sy_pnum glon ra,sy_pnum dec sy_pm,sy_pnum dec sy_plx,sy_pnum elat sy_pm,sy_pnum elat sy_plx,sy_pnum elat st_teff,sy_pnum pl_ntranspec st_rad,sy_pnum st_rad pl_orbper,...,pl_ntranspec sy_dist st_teff,disc_year st_rad pl_orbper,sy_vmag^2 st_rad,sy_vmag^2 st_mass,sy_vmag sy_bmag st_mass,sy_kmag st_teff pl_orbper,sy_gaiamag^3,st_rad sy_plx pl_orbper,st_rad st_mass soltype_Published Confirmed,st_teff^2 pl_orbper
count,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,...,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000,12280.000000
mean,-0.002332,-0.003364,0.006509,0.005918,0.005712,0.005866,0.005155,0.009258,0.000456,0.002037,...,-0.001119,-0.000201,-0.001685,0.002465,0.001864,-0.000452,0.004896,0.001578,0.002870,-0.000060
std,0.993785,0.990921,1.005047,1.009622,1.010050,1.007557,1.009808,1.009283,1.016053,0.986333,...,1.010142,0.991439,0.981667,0.996754,0.995917,0.982094,0.997933,1.002340,1.008196,0.985889
min,-2.573776,-1.302935,-1.531611,-5.346076,-4.064022,-1.753459,-1.963479,-2.223625,-0.140779,-0.371282,...,-0.126378,-0.444939,-1.448177,-3.504487,-3.595089,-0.499179,-3.192376,-0.398416,-0.306409,-0.491416
25%,-0.640573,-0.379102,-0.787916,-0.490768,-0.444684,-0.510472,-0.454819,-0.710618,-0.140779,-0.323599,...,-0.126378,-0.372286,-0.374938,-0.677812,-0.671244,-0.409211,-0.662929,-0.343065,-0.306409,-0.406474
50%,0.132709,-0.205049,-0.140324,-0.290889,-0.261435,-0.305038,-0.269590,-0.258064,-0.140779,-0.246787,...,-0.126378,-0.271529,-0.065008,0.042594,0.030475,-0.297024,0.136591,-0.263087,-0.306409,-0.296845
75%,0.132709,0.071800,0.619757,0.128351,0.074186,0.123010,0.086372,0.582668,-0.140779,-0.036777,...,-0.126378,-0.023150,0.222999,0.694685,0.683101,-0.033797,0.800862,-0.041798,-0.306409,-0.015055
max,3.999114,22.522058,7.150607,10.409760,9.941299,7.847012,10.319862,6.350118,16.110996,22.558933,...,17.457552,25.569203,31.629839,5.537985,5.852673,17.732477,2.603973,36.600556,18.752860,18.838490


In [5]:
grid = {'max_depth' : np.arange(1,20,3),'n_estimators':np.arange(1,2500,250)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1,verbose=3)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Fitting 5 folds for each of 70 candidates, totalling 350 fits
Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(19), 'n_estimators': np.int64(2001)}
    Optimal Valid R2 = 0.8195762823669698


Implementing a Heap Map to Assist Grid Search

In [6]:
Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

In [7]:
fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Refined Grid Search

In [10]:
grid = {'max_depth' : np.arange(17,21,1),'n_estimators':np.arange(1751,2250,50)}
rfr2 = RandomForestRegressor(max_features = 1/3)
rfr2CV = GridSearchCV(rfr2,param_grid= grid,n_jobs=-1)
rfr2CV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfr2CV.best_params_ )
print('    Optimal Valid R2 =', rfr2CV.best_score_ )

Scores_mean = rfr2CV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Number of Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(20), 'n_estimators': np.int64(1751)}
    Optimal Valid R2 = 0.8249643363288321


Evaluating the Model

In [25]:
#results = pd.DataFrame()
#results['trees'] = grid['n_estimators']
#results['train R2'] = rfrCV.cv_results_['mean_train_score']
#results['valid R2']  = rfrCV.cv_results_['mean_test_score']
#results[['train R2','valid R2']].plot.line()

In [12]:
# test R2
print(f" train R2 {rfr2CV.score(X_train,y_train):.3f}")
print(f" test R2 {rfr2CV.score(X_test,y_test):.3f}")

 train R2 0.965
 test R2 0.888


MSE

In [1]:
from sklearn.metrics import mean_squared_error
y_pred_test = rfr2CV.predict(X_test)
y_pred_train = rfr2CV.predict(X_train)

print(mean_squared_error(y_test,y_pred_test))
print(mean_squared_error(y_train,y_pred_train))


NameError: name 'rfr2CV' is not defined

In [ ]:
rfr_report= RandomForestRegressor(max_features = 1/3)

rfr2CV.fit(X_train,y_train)

print('Random Forest Regressor:')
print('    Optimal Parameters:', rfr2CV.best_params_ )
print('    Optimal Valid R2 =', rfr2CV.best_score_ )